## 25. Final artifact checks and robustness verdict

This final cell reloads exported results rather than relying on hidden training state. It verifies the experiment matrix and transition budgets and computes the execution-seed win rate. The primary seed was fixed in advance, but reporting it alone would conceal material variability.

In [23]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
out = Path('experiment_outputs')
runs = pd.read_csv(out / 'validation_runs.csv')
trace = pd.read_csv(out / 'training_diagnostics.csv')
execution = pd.read_csv(out / 'execution_seed_sensitivity.csv')
final = pd.read_csv(out / 'final_test_metrics.csv', index_col=0)
assert len(runs) == 39 and runs.groupby('experiment').size().eq(3).all()
assert trace.groupby(['experiment', 'seed']).steps.max().eq(24000).all()
assert np.isfinite(runs[['net_return', 'log_growth', 'max_drawdown', 'score']]).all().all()
assert runs[runs.experiment == 'E4_KL_medium'].training_updates.eq(24000).all()
assert runs[runs.experiment == 'E11_hierarchy'].training_updates.eq(4800).all()
beat = int((execution.net_return > final.loc['buy_hold', 'net_return']).sum())
verdict = pd.Series({
    'primary_path_return': final.loc['selected', 'net_return'],
    'additional_execution_median_return': execution.net_return.median(),
    'buy_hold_return': final.loc['buy_hold', 'net_return'],
    'additional_execution_paths_beating_buy_hold': beat,
    'additional_execution_paths': len(execution)
})
display(verdict.to_frame('Observed result'))
display(Markdown(f'**Final interpretation:** only **{beat}/{len(execution)}** additional execution paths beat buy-and-hold. The corrected code and validation winner do **not** establish a reliable performance improvement. Keep the baseline, the failed ablations, and the randomness results visible.'))
print('PASS: 39 runs, three seeds per configuration, equal transition budgets, finite metrics, and primitive/option update counts.')

,Observed result
primary_path_return,0.596382
additional_execution_median_return,0.287525
buy_hold_return,0.527428
additional_execution_paths_beating_buy_hold,0.000000
additional_execution_paths,20.000000


**Final interpretation:** only **0/20** additional execution paths beat buy-and-hold. The corrected code and validation winner do **not** establish a reliable performance improvement. Keep the baseline, the failed ablations, and the randomness results visible.

PASS: 39 runs, three seeds per configuration, equal transition budgets, finite metrics, and primitive/option update counts.
